# Question Generation Model Training (English + Hindi)

This notebook trains T5/mT5 models for end-to-end question generation:
- **English**: T5-small on SQuAD dataset
- **Hindi**: mT5-small on XQuAD Hindi dataset

**Make sure to enable GPU:** Runtime → Change runtime type → GPU

## 1. Clone Repository

In [ ]:
!git clone https://github.com/saiteja2873/LLM.git
%cd LLM

## 2. Install Dependencies

In [ ]:
!pip install torch transformers datasets nltk accelerate sentencepiece tqdm

## 3. Verify GPU is Available

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 4. Prepare Training Data

This downloads SQuAD dataset and processes it for question generation.

In [ ]:
!python prepare_data.py --task e2e_qg --model_type t5

## 5. Train the Model

Training with T5-small for 3 epochs. Adjust parameters as needed:
- `--model_name_or_path`: Use `t5-base` for better quality (needs more memory)
- `--num_train_epochs`: More epochs = better quality but longer training
- `--per_device_train_batch_size`: Reduce if you get OOM errors

In [ ]:
!python run_qg.py \
    --model_name_or_path t5-small \
    --model_type t5 \
    --tokenizer_name_or_path t5_qg_tokenizer \
    --output_dir ./my_model \
    --train_file_path data/train_data_e2e_qg_highlight_t5.pt \
    --valid_file_path data/valid_data_e2e_qg_highlight_t5.pt \
    --per_device_train_batch_size 8 \
    --num_train_epochs 3 \
    --learning_rate 1e-4 \
    --save_steps 5000 \
    --logging_steps 500 \
    --do_train

## 6. Test the English Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load trained English model
tokenizer = AutoTokenizer.from_pretrained("./my_model")
model = AutoModelForSeq2SeqLM.from_pretrained("./my_model")

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Test with sample text
text = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower. It was constructed from 1887 to 1889."

prompt = f"generate questions: {text}"
inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=4,
    early_stopping=True
)

questions = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated English Questions:")
print(questions)

---
# Part 2: Hindi Question Generation

Train a Hindi model using mT5 (multilingual T5) on XQuAD Hindi dataset.

## 7. Prepare Hindi Training Data

Download and process XQuAD Hindi dataset.

In [ ]:
!python prepare_hindi_data.py --method xquad --model_type mt5

## 8. Train Hindi Model

Training with mT5-small for 3 epochs on XQuAD Hindi.

In [ ]:
!python run_qg.py \
    --model_name_or_path google/mt5-small \
    --model_type mt5 \
    --tokenizer_name_or_path mt5_hindi_qg_tokenizer \
    --output_dir ./my_hindi_model \
    --train_file_path data/train_data_hindi_e2e_qg_xquad_mt5.pt \
    --valid_file_path data/valid_data_hindi_e2e_qg_xquad_mt5.pt \
    --per_device_train_batch_size 8 \
    --num_train_epochs 3 \
    --learning_rate 1e-4 \
    --save_steps 500 \
    --logging_steps 100 \
    --do_train

## 9. Test Hindi Model

In [ ]:
# Load trained Hindi model
tokenizer_hi = AutoTokenizer.from_pretrained("./my_hindi_model")
model_hi = AutoModelForSeq2SeqLM.from_pretrained("./my_hindi_model")
model_hi = model_hi.to(device)

# Test with Hindi text
text_hi = "ताजमहल भारत के आगरा शहर में यमुना नदी के किनारे स्थित है। इसका निर्माण मुगल सम्राट शाहजहाँ ने अपनी पत्नी मुमताज महल की याद में करवाया था। यह 1632 से 1653 के बीच बनाया गया था।"

prompt_hi = f"प्रश्न उत्पन्न करें: {text_hi}"
inputs_hi = tokenizer_hi(prompt_hi, return_tensors="pt", max_length=512, truncation=True).to(device)

outputs_hi = model_hi.generate(
    **inputs_hi,
    max_length=128,
    num_beams=4,
    early_stopping=True
)

questions_hi = tokenizer_hi.decode(outputs_hi[0], skip_special_tokens=True)
print("Generated Hindi Questions (हिंदी प्रश्न):")
print(questions_hi)

## 10. Download Both Models

Download both English and Hindi models to use locally.

In [ ]:
# Zip both models
!zip -r my_model_english.zip my_model/
!zip -r my_model_hindi.zip my_hindi_model/

from google.colab import files

# Download English model
print("Downloading English model...")
files.download('my_model_english.zip')

# Download Hindi model  
print("Downloading Hindi model...")
files.download('my_model_hindi.zip')

## 11. (Optional) Push to Hugging Face Hub

Share your models on Hugging Face.

In [ ]:
# Uncomment and run to push to Hugging Face
# from huggingface_hub import login
# login()  # Enter your HF token

# Push English model
# model.push_to_hub("your-username/question-generation-t5-english")
# tokenizer.push_to_hub("your-username/question-generation-t5-english")

# Push Hindi model
# model_hi.push_to_hub("your-username/question-generation-mt5-hindi")
# tokenizer_hi.push_to_hub("your-username/question-generation-mt5-hindi")